In [91]:
import pickle
import numpy as np
from pathlib import Path
import pandas as pd 
import librosa 
import IPython.display as ipd
import soundfile as sf


### Re-cut SWC stimuli for better excerpts in binaural experiment 

In [70]:
path_to_dir = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2024')
path_to_dir.mkdir(exist_ok = True, parents=True)


In [2]:
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 

In [3]:
word_list = list(class_map.keys())

In [4]:
with open("word_list_for_binaural_sentences.pkl", "rb") as f:
	target_word_list =  pickle.load(f)

### First, re-screen SWC for excerpts in this list of target words

In [25]:
swc_fname = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'
swc_main_df = pd.read_pickle(swc_fname)
print(swc_main_df.shape)

(1006299, 13)


In [36]:
## Throw out bad talkers

bad_talker_names = ['wikipedia', 'viktor-o-ledenyov',  'acerperi']

In [37]:
swc_main_df = swc_main_df[~swc_main_df.client_id.str.contains('|'.join(bad_talker_names))]

In [63]:
### remove examples that end too close to edge of file 

tol_in_s = .5 
swc_main_df = swc_main_df[swc_main_df.clip_end_in_s <= (swc_main_df.total_file_duration_in_s - tol_in_s)]


In [64]:
swc_main_df.shape

(992956, 13)

In [75]:
pd.__version__

'1.4.2'

In [76]:
gender_balanced_swc = swc_main_df[swc_main_df.word.isin(target_word_list)].groupby(['gender', 'word']).sample(1, replace=False, random_state=0).reset_index()

In [81]:
#rename index
gender_balanced_swc.rename(columns={'index':'orig_df_ix'}, inplace=True)

In [82]:
### Write parent manifest to dir 
out_name = path_to_dir / 'test_stim_orig_manifest.pdpkl'
gender_balanced_swc.to_pickle(out_name)

In [83]:
gender_balanced_swc.head()

,orig_df_ix,client_id,clip_dur_in_s,clip_end_in_s,clip_start_in_s,corpus,gender,gender_int,split,split_int,sr,src_fn,total_file_duration_in_s,word
0,601538,laura-s,0.29,1205.07,1204.78,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,1796.176689,about
1,638828,dolliellama,0.36,737.92,737.56,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,4871.649524,above
2,249863,ama1016,0.52,371.41,370.89,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,488.501406,access
3,127418,popularoutcast,0.47,3399.30,3398.83,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,3717.672925,across
4,961747,sedola,0.42,1898.92,1898.50,swc,female,0,NaN,0,44100,/scratch2/weka/mcdermott/msaddler/swc/english/...,2335.023311,action


In [90]:
# get manifest for cue and distractor excerpts 

bg_df = swc_main_df[(~swc_main_df.index.isin(gender_balanced_swc.orig_df_ix)) & (swc_main_df.word.isin(word_list))].reset_index()
bg_df.rename(columns={'index':'orig_df_ix'}, inplace=True)
print(bg_df.shape)
# get partition of swc

(317083, 14)


In [104]:
### For row target in gender_balanced_swc load 3 seconds from the src_fn centered on the clip_start_in_s and clip_end_in_s

total_dur = 3 # in seconds
human_dur = 2 # in seconds 
SR = 44_100

def get_window_bounds(clip_dur_in_s, clip_start_in_s, clip_end_in_s, total_dur):
	
	offset = (total_dur  - clip_dur_in_s) / 2
	return clip_start_in_s - offset, clip_end_in_s + offset
# given clip_dur_in_s find lag and lead time around clip_end_in_s and clip_start_in_s

row = gender_balanced_swc.iloc[10]
print(row)
src_fn = row['src_fn']
clip_start_in_s = row['clip_start_in_s']
clip_end_in_s = row['clip_end_in_s']
clip_dur_in_s = row['clip_dur_in_s']
word = row['word']
onset, offset = get_window_bounds(clip_dur_in_s, clip_start_in_s, clip_end_in_s, total_dur)
print(onset, offset)
wav, sr = librosa.load(src_fn, sr=SR, offset=onset, duration=total_dur, dtype=np.float32)
print(word)
ipd.Audio(wav, rate=sr, normalize=False)	




orig_df_ix                                                             380822
client_id                                                           laurahale
clip_dur_in_s                                                             0.3
clip_end_in_s                                                           50.13
clip_start_in_s                                                         49.83
corpus                                                                    swc
gender                                                                 female
gender_int                                                                  0
split                                                                     NaN
split_int                                                                   0
sr                                                                      44100
src_fn                      /scratch2/weka/mcdermott/msaddler/swc/english/...
total_file_duration_in_s                                        